# Dataset Choice and preliminary - Regressions
Task 1 - Task 2
**Dataset:** [Melbourne Housing Snapshot](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot)
Target: `Price`

In [ ]:
import sys, glob, warnings

warnings.filterwarnings("ignore")
sys.path.append("../../")  # lets us import helper.py
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from helper import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_FILE = glob.glob("../../data/melbourne/*.csv")[0]
TARGET = "Price"
df = pd.read_csv(DATA_FILE)


df["Date"] = pd.to_datetime(
    df["Date"], dayfirst=True, errors="coerce"
)  # sale date: text - datetime
df["Postcode"] = (
    df["Postcode"].astype("Int64").astype(str)
)  # a code, not a quantity - text
LAT = next(
    c for c in df.columns if c.lower().startswith("lat")
)  # the Kaggle file spells it "Lattitude"
LON = next(c for c in df.columns if c.lower().startswith("long"))

num_cols, cat_cols = split_columns(
    df, TARGET
)  # feature lists WITHOUT the target (Date is excluded from both)
print(
    df.shape,
    "| numeric features:",
    len(num_cols),
    "| categorical features:",
    len(cat_cols),
)

## 1a) Datatypes

In [ ]:
df.info()
display(column_report(df))
display(df.head())

## 1b) Shapes

In [ ]:
n_rows, n_cols = df.shape
levels = {c: df[c].nunique() for c in cat_cols}
print(f"Rows: {n_rows} | Columns: {n_cols}")
print(
    "Columns you would get after one-hot encoding every text column:",
    len(num_cols) + sum(levels.values()),
)
print(f"Rows per raw feature: {n_rows/(n_cols-1):.0f}")
pd.Series(levels).sort_values().plot.barh(figsize=(8, 4), color="steelblue")
plt.title("Distinct values per text column")
plt.show()

## 1c) Missing values, diblicates, zeros, impossible values

In [ ]:
miss = column_report(df)[["missing", "missing_%"]].sort_values(
    "missing", ascending=False
)
display(miss[miss["missing"] > 0])
plot_missing(df)
print("Missing TARGET values:", int(df[TARGET].isna().sum()))
print("Complete duplicate rows:", int(df.duplicated().sum()))
print("Same Address + Date duplicated:", int(df.duplicated(["Address", "Date"]).sum()))

In [ ]:
zero_cols = [
    "Landsize",
    "BuildingArea",
    "Bathroom",
    "Car",
    "Distance",
    "Rooms",
    "Bedroom2",
]
display(
    pd.DataFrame(
        {
            "zeros": (df[zero_cols] == 0).sum(),
            "zeros_%": ((df[zero_cols] == 0).mean() * 100).round(2),
        }
    )
)

sale_year = df["Date"].dt.year
checks = pd.Series(
    {
        "Price <= 0": (df[TARGET] <= 0).sum(),
        "Landsize == 0": (df["Landsize"] == 0).sum(),
        "BuildingArea == 0": (df["BuildingArea"] == 0).sum(),
        "Bathroom == 0": (df["Bathroom"] == 0).sum(),
        "Bedroom2 != Rooms": (df["Bedroom2"] != df["Rooms"]).sum(),
        "YearBuilt < 1800": (df["YearBuilt"] < 1800).sum(),
        "YearBuilt after sale year": (df["YearBuilt"] > sale_year).sum(),
        "BuildingArea > Landsize (land > 0)": (
            (df["BuildingArea"] > df["Landsize"]) & (df["Landsize"] > 0)
        ).sum(),
        "Coordinates outside Melbourne box": (
            ~df[LAT].between(-38.6, -37.2) | ~df[LON].between(144.3, 145.9)
        ).sum(),
        "Date could not be parsed": df["Date"].isna().sum(),
    }
).to_frame("rows")
display(checks)
print("Sale dates:", df["Date"].min().date(), "->", df["Date"].max().date())
display(df[num_cols].describe().T)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(df["YearBuilt"].dropna(), bins=60, ax=ax[0])
ax[0].set_title("YearBuilt (look at the far left)")
sns.histplot(df["Landsize"], bins=60, ax=ax[1])
ax[1].set_title("Landsize")
sns.scatterplot(data=df, x="BuildingArea", y="Landsize", alpha=0.3, ax=ax[2])
ax[2].set_title("BuildingArea vs Landsize")
plt.tight_layout()
plt.show()

## 1d) Alerts from big-picture tools

In [ ]:
import os

os.makedirs("../../reports", exist_ok=True)
try:
    from ydata_profiling import ProfileReport

    profile = ProfileReport(
        df, title="Melbourne Housing - preliminary EDA", explorative=True
    )
    profile.to_file("../../reports/melbourne_ydata_profile.html")
    print("Saved reports/melbourne_ydata_profile.html  -> open it in your browser")
    try:
        display(
            pd.DataFrame(
                {
                    "ydata-profiling alert": [
                        str(a) for a in profile.get_description().alerts
                    ]
                }
            )
        )
    except Exception as e:
        print("(Could not list alerts here - read the Alerts tab in the HTML.)", e)
except ImportError:
    print("ydata-profiling is not installed.  Run:  pip install ydata-profiling")

In [ ]:
try:
    import sweetviz as sv

    sv.analyze(df.drop(columns=["Address"]), target_feat=TARGET).show_html(
        "../../reports/melbourne_sweetviz.html", open_browser=False
    )
    print("Saved reports/melbourne_sweetviz.html")
except Exception as e:
    print("SweetViz skipped:", type(e).__name__, "- optional: pip install sweetviz")

## personal notes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(
    df[num_cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    annot_kws={"size": 7},
    ax=ax[0],
)
ax[0].set_title("Correlation (numeric)")
sns.histplot(df[TARGET].dropna(), bins=60, kde=True, ax=ax[1])
ax[1].set_title(f"{TARGET} (skew = {df[TARGET].skew():.2f})")
plt.tight_layout()
plt.show()
display(high_corr_pairs(df, num_cols, 0.7))